<a href="https://colab.research.google.com/github/darshangate10-ctrl/DL-Paper-Implementation-/blob/main/CODE_CLONE_DETECTION_SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code Clone Detection System
## Multi-View Code Representation Learning Framework

> **Theoretical Grounding**:  
> *Hazem Peter Samoaa, Firas Bayram, Pasquale Salza, and Philipp Leitner.*  
> **"A systematic mapping study of source code representation for deep learning in software engineering."**  
> *IET Software, Volume 16, Issue 4, pp. 351–385, 2022.*

---

### Dual-Engine Architecture Overview
```text
                 CODE CLONE DETECTION SYSTEM
                             │
             ┌───────────────┴───────────────┐
             ↓                               ↓
       CODE-TO-CODE                  CODE-TO-PREDICTION
             │                               │
      Code A + Code B                 Code A + Code B
             ↓                               ↓
      Code Comparison                   Trained ML Model
             ↓                               ↓
   Similarity / Differences            CLONE / NON-CLONE
   (Token + AST Structure)         (Hybrid Siamese Network)
```

### Purpose & Difference Between the Two Features
1. **CODE-TO-CODE**: Focuses on **source-code content and AST structure comparison**. Computes token sequence similarity, syntax tree edit similarity, and highlights matching vs. differing code lines without naming bias.
2. **CODE-TO-PREDICTION**: Focuses on **deep learning classification**. Uses a trained Siamese Neural Network with Token BiLSTM, Tree AST-BiLSTM, and Graph GCN encoders with cross-modality attention fusion to classify whether two programs are semantically equivalent clones (`CLONE` vs `NON-CLONE`).


## 1. System Setup & Imports
We import standard dependencies (`torch`, `numpy`, `scikit-learn`) and configure reproducible seeds.

> **Google Colab Note**: If running for the first time, click **Runtime -> Run all** (or press `Ctrl+F9`) to initialize all cells in order.


In [ ]:
# -------------------------------------------------------------
# Colab / Environment Setup & Safe Imports
# -------------------------------------------------------------
try:
    import google.colab
    !pip install -q torch numpy scikit-learn
except Exception:
    pass

from __future__ import annotations
import os
import sys
import re
import ast
import random
from typing import List, Dict, Tuple, Set, Any, Optional

sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def setup_device(seed: int = 42) -> torch.device:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f'[Hardware] Using GPU: {torch.cuda.get_device_name(0)}')
    else:
        device = torch.device('cpu')
        print('[Hardware] Using CPU (Colab-Safe Mode)')
    return device

device = setup_device()
print('PyTorch Version:', torch.__version__)


[Hardware] Using GPU: Tesla T4
PyTorch Version: 2.11.0+cu128


## 2. Multi-View Source Code Feature Extractors
Code snippets are abstracted into three complementary modalities:
* **Token View**: Lexical streams with identifier sub-token decomposition and literal normalization.
* **Tree View (AST)**: Linearized via **Structure-Based Traversal (SBT)** preserving grammar hierarchy.
* **Graph View**: Combined **Control Flow Graph (CFG)** and **Data Flow Graph (DFG)** with normalized adjacency.


In [1]:
from __future__ import annotations
import os
import sys
import re
import ast
import random
from typing import List, Dict, Tuple, Set, Any, Optional
import numpy as np

# -------------------------------------------------------------
# 2.1 Lexical Tokenizer with Sub-word Decomposition
# -------------------------------------------------------------
class CodeTokenizer:
    KEYWORDS = {
        'def', 'return', 'if', 'else', 'elif', 'for', 'while', 'in', 'range',
        'int', 'float', 'str', 'bool', 'True', 'False', 'None', 'and', 'or',
        'not', 'break', 'continue', 'pass', 'len', 'print'
    }
    OPERATORS = {
        '+', '-', '*', '/', '%', '==', '!=', '<', '>', '<=', '>=', '=',
        '+=', '-=', '*=', '/=', '(', ')', '[', ']', '{', '}', ',', ':', ';'
    }

    def __init__(self, max_vocab_size: int = 500):
        self.token2idx = {'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3}
        self.idx2token = {v: k for k, v in self.token2idx.items()}
        self.max_vocab_size = max_vocab_size

    @staticmethod
    def split_words(name: str) -> List[str]:
        spaced = re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', name)
        words = spaced.replace('_', ' ').lower().split()
        return words if words else [name.lower()]

    def tokenize(self, code: str) -> List[str]:
        lines = [line.split('#')[0] for line in code.strip().split('\n')]
        clean = ' '.join(lines)
        pattern = re.compile(r'([a-zA-Z_]\w*|\d+\.?\d*|==|!=|<=|>=|\+=|-=|\*=|/=|->|[+\-*/%<>=()\[\]{},:;.])')
        tokens = []
        for item in pattern.findall(clean):
            if item in self.KEYWORDS or item in self.OPERATORS:
                tokens.append(item)
            elif re.match(r'^\d', item):
                tokens.append('<NUM>')
            else:
                tokens.extend(self.split_words(item))
        return tokens

    def encode(self, code: str, max_len: int = 64) -> List[int]:
        tokens = self.tokenize(code)
        ids = [self.token2idx['<BOS>']]
        for t in tokens[:max_len - 2]:
            ids.append(self.token2idx.get(t, self.token2idx['<UNK>']))
        ids.append(self.token2idx['<EOS>'])
        if len(ids) < max_len: ids.extend([self.token2idx['<PAD>']] * (max_len - len(ids)))
        return ids[:max_len]

# -------------------------------------------------------------
# 2.2 AST Structure-Based Traversal (SBT)
# -------------------------------------------------------------
class ASTParser:
    def __init__(self, max_vocab_size: int = 500):
        self.node2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2node = {0: '<PAD>', 1: '<UNK>'}
        self.max_vocab_size = max_vocab_size

    def _traverse_sbt(self, node: Any, traversal: List[str]):
        if node is None: return
        name = type(node).__name__
        traversal.append(f'({name}')
        if isinstance(node, ast.Name): traversal.append(f'id_{node.id.lower()}')
        elif isinstance(node, ast.Constant): traversal.append(f'const_{type(node.value).__name__}')
        elif isinstance(node, ast.arg): traversal.append(f'arg_{node.arg.lower()}')
        for child in ast.iter_child_nodes(node):
            self._traverse_sbt(child, traversal)
        traversal.append(f'){name}')

    def parse_sbt(self, code: str) -> List[str]:
        try:
            tree = ast.parse(code)
            tokens = []
            self._traverse_sbt(tree, tokens)
            return tokens
        except Exception:
            return ['(Module', '(FunctionDef', ')FunctionDef', ')Module']

    def encode(self, code: str, max_len: int = 96) -> List[int]:
        tokens = self.parse_sbt(code)
        ids = [self.node2idx.get(t, self.node2idx['<UNK>']) for t in tokens[:max_len]]
        if len(ids) < max_len: ids.extend([self.node2idx['<PAD>']] * (max_len - len(ids)))
        return ids[:max_len]

# -------------------------------------------------------------
# 2.3 Program Graph Builder (CFG + DFG)
# -------------------------------------------------------------
class ProgramGraphBuilder:
    def __init__(self, max_nodes: int = 20, node_feat_dim: int = 16):
        self.max_nodes = max_nodes
        self.node_feat_dim = node_feat_dim
        self.stmt_types = {'Entry': 0, 'FunctionDef': 1, 'Assign': 2, 'AugAssign': 3, 'For': 4, 'While': 5, 'If': 6, 'Return': 7, 'Expr': 8, 'Exit': 9}

    def build_graph(self, code: str) -> Tuple[np.ndarray, np.ndarray]:
        nodes = [{'type': 'Entry', 'defs': set(), 'uses': set()}]
        control_edges, data_edges = [], []
        try:
            tree = ast.parse(code)
            stmts = [n for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.Assign, ast.AugAssign, ast.For, ast.While, ast.If, ast.Return, ast.Expr))]
            for s in stmts[:self.max_nodes - 2]:
                defs, uses = set(), set()
                for c in ast.walk(s):
                    if isinstance(c, ast.Name):
                        if isinstance(c.ctx, ast.Store): defs.add(c.id)
                        elif isinstance(c.ctx, ast.Load): uses.add(c.id)
                nodes.append({'type': type(s).__name__, 'defs': defs, 'uses': uses})
            for i in range(len(nodes) - 1): control_edges.append((i, i + 1))
            last_def = {}
            for idx, n in enumerate(nodes):
                for u in n['uses']:
                    if u in last_def: data_edges.append((last_def[u], idx))
                for d in n['defs']: last_def[d] = idx
        except Exception:
            nodes = [{'type': 'Entry'}, {'type': 'FunctionDef'}]
            control_edges = [(0, 1)]
        if len(nodes) < self.max_nodes: nodes.append({'type': 'Exit'})
        num_nodes = min(len(nodes), self.max_nodes)
        node_feats = np.zeros((self.max_nodes, self.node_feat_dim), dtype=np.float32)
        for i in range(num_nodes):
            t_idx = self.stmt_types.get(nodes[i].get('type', 'Expr'), 0)
            if t_idx < self.node_feat_dim: node_feats[i, t_idx] = 1.0
        adj = np.eye(self.max_nodes, dtype=np.float32)
        for u, v in control_edges + data_edges:
            if u < self.max_nodes and v < self.max_nodes:
                adj[u, v] = 1.0; adj[v, u] = 0.5
        deg = np.sum(adj, axis=1)
        deg_inv = np.power(np.maximum(deg, 1e-6), -0.5)
        norm_adj = np.diag(deg_inv) @ adj @ np.diag(deg_inv)
        return node_feats, norm_adj.astype(np.float32)

print('Feature extractors ready.')


Feature extractors ready.


## 3. Deep Learning Hybrid Architecture
Preserves the exact neural architecture:
* **TokenBiLSTM**: Bi-LSTM with attention pooling.
* **TreeASTBiLSTM**: SBT sequence encoder with multi-scale 1D convolutions.
* **GraphGCN**: Vectorized 2-layer Graph Convolutional Network.
* **AttentionFusion**: Cross-modality attention weighting over the three representations.
* **SiameseCloneDetector**: Siamese comparison with interaction features $[h_A, h_B, |h_A - h_B|, h_A \odot h_B, \text{sim}]$.


In [2]:
from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F

class TokenBiLSTM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 64, hidden_dim: int = 64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim // 2, bidirectional=True, batch_first=True)
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        weights = F.softmax(self.attn(out), dim=1)
        return torch.sum(out * weights, dim=1)

class TreeASTBiLSTM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 64, hidden_dim: int = 64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim // 2, bidirectional=True, batch_first=True)
        self.conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1)

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        conv_out = F.relu(self.conv(lstm_out.transpose(1, 2))).transpose(1, 2)
        pooled, _ = torch.max(conv_out, dim=1)
        return pooled

class GraphGCN(nn.Module):
    def __init__(self, in_dim: int = 16, hidden_dim: int = 64):
        super().__init__()
        self.linear1 = nn.Linear(in_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, node_feats, adj):
        h1 = F.relu(torch.bmm(adj, self.linear1(node_feats)))
        h2 = F.relu(torch.bmm(adj, self.linear2(h1)))
        return torch.mean(h2, dim=1)

class AttentionFusion(nn.Module):
    def __init__(self, hidden_dim: int = 64):
        super().__init__()
        self.attn_layer = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1))
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, h_tok, h_tree, h_graph):
        views = torch.stack([h_tok, h_tree, h_graph], dim=1)
        weights = F.softmax(self.attn_layer(views), dim=1)
        fused = torch.sum(views * weights, dim=1)
        return self.proj(fused), weights.squeeze(-1)

class HybridCodeEncoder(nn.Module):
    def __init__(self, tok_vocab: int, ast_vocab: int, graph_dim: int = 16, hidden_dim: int = 64):
        super().__init__()
        self.tok_enc = TokenBiLSTM(tok_vocab, hidden_dim=hidden_dim)
        self.tree_enc = TreeASTBiLSTM(ast_vocab, hidden_dim=hidden_dim)
        self.graph_enc = GraphGCN(graph_dim, hidden_dim=hidden_dim)
        self.fusion = AttentionFusion(hidden_dim=hidden_dim)

    def forward(self, tok, ast_seq, node_feats, adj):
        h_tok = self.tok_enc(tok)
        h_tree = self.tree_enc(ast_seq)
        h_graph = self.graph_enc(node_feats, adj)
        return self.fusion(h_tok, h_tree, h_graph)

class SiameseCloneDetector(nn.Module):
    def __init__(self, encoder: HybridCodeEncoder, hidden_dim: int = 64):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, b_a, b_b):
        h_a, attn_a = self.encoder(b_a['token'], b_a['ast'], b_a['node_feats'], b_a['adj'])
        h_b, attn_b = self.encoder(b_b['token'], b_b['ast'], b_b['node_feats'], b_b['adj'])
        h_a_norm = F.normalize(h_a, p=2, dim=-1)
        h_b_norm = F.normalize(h_b, p=2, dim=-1)
        diff = torch.abs(h_a_norm - h_b_norm)
        prod = h_a_norm * h_b_norm
        cos_sim = torch.sum(prod, dim=-1, keepdim=True)
        comp = torch.cat([h_a_norm, h_b_norm, diff, prod, cos_sim], dim=-1)
        logits = self.classifier(comp).squeeze(-1)
        return logits, attn_a, attn_b

print('Neural network architecture loaded.')


Neural network architecture loaded.


## 4. Pre-Trained Model Loading (Loaded Once)
We load the trained model weights (`saved_models/siamese_clone_detector.pt`) and serialized vocabularies (`saved_models/vocabs.pt`).


In [3]:
from __future__ import annotations
import os
import sys
import torch
sys.path.insert(0, os.path.abspath('.'))

ckpt_path = 'saved_models/siamese_clone_detector.pt'
vocab_path = 'saved_models/vocabs.pt'

if os.path.exists(ckpt_path) and os.path.exists(vocab_path):
    vocab_data = torch.load(vocab_path, map_location='cpu', weights_only=False)
    tokenizer = CodeTokenizer()
    tokenizer.token2idx = vocab_data['token2idx']
    tokenizer.idx2token = vocab_data['idx2token']

    ast_parser = ASTParser()
    ast_parser.node2idx = vocab_data['node2idx']
    ast_parser.idx2node = vocab_data['idx2node']

    graph_builder = ProgramGraphBuilder()

    encoder = HybridCodeEncoder(len(tokenizer.token2idx), len(ast_parser.node2idx), graph_dim=16, hidden_dim=64)
    model = SiameseCloneDetector(encoder, hidden_dim=64).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f'[ModelManager] Model checkpoint successfully loaded from {ckpt_path}!')
else:
    print('[Warning] Checkpoint not found; run test_services.py or run_server.py to create it.')


[Warning] Checkpoint not found; run test_services.py or run_server.py to create it.


---
# FEATURE 1: CODE-TO-CODE
### Workflow
```text
Code A + Code B  ──>  Code Comparison  ──>  Similarity Score / Differences
```
### Purpose
Compare two source code files for **lexical token similarity**, **AST structural similarity**, and **statement-level differences** without false bias from renamed variables.


In [5]:
import os
import sys

# --- BEGIN FIX: Implement actual backend module and service ---
!mkdir -p backend
!echo "" > backend/__init__.py

# Use Python to write the file content
with open('backend/compare_service.py', 'w') as f:
    f.write('''
import re
import ast
from typing import List, Dict, Tuple, Set, Any, Optional

# -------------------------------------------------------------
# 2.1 Lexical Tokenizer with Sub-word Decomposition
# (Copied from cell HL6kgcqp9bZH for self-contained module)
# -------------------------------------------------------------
class CodeTokenizer:
    KEYWORDS = {
        'def', 'return', 'if', 'else', 'elif', 'for', 'while', 'in', 'range',
        'int', 'float', 'str', 'bool', 'True', 'False', 'None', 'and', 'or',
        'not', 'break', 'continue', 'pass', 'len', 'print'
    }
    OPERATORS = {
        '+', '-', '*', '/', '%', '==', '!=', '<', '>', '<=', '>=', '=',
        '+=', '-=', '*=', '/=', '(', ')', '[', ']', '{', '}', ',', ':', ';'}

    def __init__(self, max_vocab_size: int = 500):
        # Dummy vocab, actual vocab building not needed for comparison via sets
        self.token2idx = {'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3}
        self.idx2token = {v: k for k, v in self.token2idx.items()}
        self.max_vocab_size = max_vocab_size

    @staticmethod
    def split_words(name: str) -> List[str]:
        spaced = re.sub(r'([a-z0-9])([A-Z])', r'\\1 \\2', name)
        words = spaced.replace('_', ' ').lower().split()
        return words if words else [name.lower()]

    def tokenize(self, code: str) -> List[str]:
        lines = [line.split('#')[0] for line in code.strip().split('\\n')]
        clean = ' '.join(lines)
        pattern = re.compile(r'([a-zA-Z_]\\w*|\\d+\\.?\\d*|==|!=|<=|>=|\\+=|-=|\\*=|/=|->|[+\\-\\*/%<>=()\\[\\]{},:;\\.])')
        tokens = []
        for item in pattern.findall(clean):
            if item in self.KEYWORDS or item in self.OPERATORS:
                tokens.append(item)
            elif re.match(r'^\\d', item):
                tokens.append('<NUM>')
            else:
                tokens.extend(self.split_words(item))
        return tokens

    def encode(self, code: str, max_len: int = 64) -> List[int]:
        # This method is not directly used by CodeComparator.compare, but part of the class definition.
        tokens = self.tokenize(code)
        ids = [self.token2idx['<BOS>']]
        for t in tokens[:max_len - 2]:
            ids.append(self.token2idx.get(t, self.token2idx['<UNK>']))
        ids.append(self.token2idx['<EOS>'])
        if len(ids) < max_len: ids.extend([self.token2idx['<PAD>']] * (max_len - len(ids)))
        return ids[:max_len]

# -------------------------------------------------------------
# 2.2 AST Structure-Based Traversal (SBT)
# (Copied from cell HL6kgcqp9bZH for self-contained module)
# -------------------------------------------------------------
class ASTParser:
    def __init__(self, max_vocab_size: int = 500):
        # Dummy vocab, actual vocab building not needed for comparison via sets
        self.node2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2node = {0: '<PAD>', 1: '<UNK>'}
        self.max_vocab_size = max_vocab_size

    def _traverse_sbt(self, node: Any, traversal: List[str]):
        if node is None: return
        name = type(node).__name__
        traversal.append(f'({name}')
        if isinstance(node, ast.Name): traversal.append(f'id_{node.id.lower()}')
        elif isinstance(node, ast.Constant): traversal.append(f'const_{type(node.value).__name__}')
        elif isinstance(node, ast.arg): traversal.append(f'arg_{node.arg.lower()}')
        for child in ast.iter_child_nodes(node):
            self._traverse_sbt(child, traversal)
        traversal.append(f'){name}')

    def parse_sbt(self, code: str) -> List[str]:
        try:
            tree = ast.parse(code)
            tokens = []
            self._traverse_sbt(tree, tokens)
            return tokens
        except Exception:
            # Fallback for unparseable code (e.g., incomplete snippets)
            return ['(Module', '(FunctionDef', ')FunctionDef', ')Module']

    def encode(self, code: str, max_len: int = 96) -> List[int]:
        # This method is not directly used by CodeComparator.compare, but part of the class definition.
        tokens = self.parse_sbt(code)
        ids = [self.node2idx.get(t, self.node2idx['<UNK>']) for t in tokens[:max_len]]
        if len(ids) < max_len: ids.extend([self.node2idx['<PAD>']] * (max_len - len(ids)))
        return ids[:max_len]


class CodeComparator:
    @staticmethod
    def compare(code_a: str, code_b: str, language: str = 'python') -> Dict[str, Any]:
        # For this implementation, we assume language is python and ignore it for now.
        tokenizer = CodeTokenizer()
        ast_parser = ASTParser()

        # Token-level comparison
        tokens_a = tokenizer.tokenize(code_a)
        tokens_b = tokenizer.tokenize(code_b)

        set_tokens_a = set(tokens_a)
        set_tokens_b = set(tokens_b)

        token_overlap = len(set_tokens_a.intersection(set_tokens_b))
        token_union = len(set_tokens_a.union(set_tokens_b))

        token_similarity = (token_overlap / token_union * 100) if token_union > 0 else 0.0

        # AST structural comparison
        sbt_a = ast_parser.parse_sbt(code_a)
        sbt_b = ast_parser.parse_sbt(code_b)

        set_sbt_a = set(sbt_a)
        set_sbt_b = set(sbt_b)

        ast_overlap = len(set_sbt_a.intersection(set_sbt_b))
        ast_union = len(set_sbt_a.union(set_sbt_b))

        structural_ast_similarity = (ast_overlap / ast_union * 100) if ast_union > 0 else 0.0

        # Overall similarity (simple average for now)
        code_similarity_percentage = (token_similarity + structural_ast_similarity) / 2.0

        # Line-by-line comparison for matching/different sections
        lines_a = code_a.strip().split('\\n')
        lines_b = code_b.strip().split('\\n')

        matching_sections = []
        different_sections = []

        max_len_lines = max(len(lines_a), len(lines_b))
        for i in range(max_len_lines):
            line_a_content = lines_a[i].strip() if i < len(lines_a) else ''
            line_b_content = lines_b[i].strip() if i < len(lines_b) else ''

            if line_a_content == line_b_content and line_a_content != '':
                matching_sections.append({'line_a': i + 1, 'line_b': i + 1, 'content': line_a_content})
            elif line_a_content != '' or line_b_content != '':
                diff_entry = {}
                if i < len(lines_a):
                    diff_entry['line_a'] = i + 1
                    diff_entry['content_a'] = line_a_content
                else:
                    diff_entry['line_a'] = None
                    diff_entry['content_a'] = ''

                if i < len(lines_b):
                    diff_entry['line_b'] = i + 1
                    diff_entry['content_b'] = line_b_content
                else:
                    diff_entry['line_b'] = None
                    diff_entry['content_b'] = ''

                different_sections.append(diff_entry)


        result_message = 'Identical' if code_similarity_percentage == 100.0 else (
                         'Highly Similar' if code_similarity_percentage >= 80.0 else (
                         'Similar' if code_similarity_percentage >= 50.0 else 'Different'))

        return {
            'code_similarity_percentage': code_similarity_percentage,
            'result': result_message,
            'metrics': {
                'token_similarity': token_similarity,
                'structural_ast_similarity': structural_ast_similarity
            },
            'matching_sections': matching_sections,
            'different_sections': different_sections
        }
''')
# --- END FIX ---

# --- BEGIN FIX: Force module reload ---
if 'backend.compare_service' in sys.modules:
    del sys.modules['backend.compare_service']
if 'backend' in sys.modules:
    del sys.modules['backend']
# --- END FIX ---

from __future__ import annotations
sys.path.insert(0, os.path.abspath('.'))
from backend.compare_service import CodeComparator

# Input Code A: Standard Bubble Sort
code_a = '''def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr'''

# Input Code B: Renamed Variables Bubble Sort
code_b = '''def sort_items(lst):
    size = len(lst)
    for p in range(size):
        for q in range(0, size - p - 1):
            if lst[q] > lst[q + 1]:
                lst[q], lst[q + 1] = lst[q + 1], lst[q]
    return lst'''

# Run Code-to-Code Comparison
c2c_result = CodeComparator.compare(code_a, code_b, language='python')

print('=== CODE-TO-CODE COMPARISON OUTPUT ===')
print(f'Overall Similarity Score:  {c2c_result["code_similarity_percentage"]:0.1f}%')
print(f'Comparison Result:         {c2c_result["result"]}')
print(f'Token-Level Similarity:    {c2c_result["metrics"]["token_similarity"]:0.1f}%')
print(f'Structural AST Similarity: {c2c_result["metrics"]["structural_ast_similarity"]:0.1f}%')
print(f'Matching Functional Blocks: {len(c2c_result["matching_sections"])}')
print(f'Different Lines Highlighted: {len(c2c_result["different_sections"])}')

=== CODE-TO-CODE COMPARISON OUTPUT ===
Overall Similarity Score:  73.3%
Comparison Result:         Similar
Token-Level Similarity:    65.5%
Structural AST Similarity: 81.1%
Matching Functional Blocks: 0
Different Lines Highlighted: 7


---
# FEATURE 2: CODE-TO-PREDICTION
### Workflow
```text
Code A + Code B  ──>  Trained Deep Learning Model  ──>  CLONE / NON-CLONE
```
### Purpose
Evaluates whether two programs are semantically equivalent clones using the pre-loaded **Multi-View Siamese Neural Network**.
Outputs the dynamic attention breakdown: **Token View**, **Tree/AST View**, and **Graph View** computed directly by the model.


In [7]:
from __future__ import annotations
import os
import sys
sys.path.insert(0, os.path.abspath('.'))
from backend.model_service import ModelManager

mgr = ModelManager.get_instance()
mgr.initialize_and_load()

# Test 1: Known Clone Pair (Factorial: Iterative Loop vs Recursion)
factorial_iter = '''def factorial(n):
    res = 1
    for i in range(1, n + 1):
        res = res * i
    return res'''

factorial_rec = '''def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)'''

# Test 2: Known Non-Clone Pair (Bubble Sort vs Primality Test)
prime_checker = '''def is_prime(n):
    if n <= 1: return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0: return False
    return True'''

pred_clone = mgr.predict(factorial_iter, factorial_rec)
pred_non_clone = mgr.predict(code_a, prime_checker)

print('=== PREDICTION 1: CLONE PAIR (Factorial Loop vs Recursion) ===')
print(f'Prediction:   {pred_clone["prediction"]}')
print(f'Probability:  {pred_clone["probability"]*100:.1f}%')
print(f'Similarity:   {pred_clone["similarity"]*100:.1f}%')
print('Dynamic Attention Weights (From Trained Model):')
print(f'  • Token View:    {pred_clone["weights"]["token_view"]:.1f}%')
print(f'  • Tree/AST View: {pred_clone["weights"]["tree_ast_view"]:.1f}%')
print(f'  • Graph View:    {pred_clone["weights"]["graph_cfg_dfg_view"]:.1f}%')

print('\n=== PREDICTION 2: NON-CLONE PAIR (Bubble Sort vs Prime Checker) ===')
print(f'Prediction:   {pred_non_clone["prediction"]}')
print(f'Probability:  {pred_non_clone["probability"]*100:.1f}%')
print(f'Similarity:   {pred_non_clone["similarity"]*100:.1f}%')
print('Dynamic Attention Weights (From Trained Model):')
print(f'  • Token View:    {pred_non_clone["weights"]["token_view"]:.1f}%')
print(f'  • Tree/AST View: {pred_non_clone["weights"]["tree_ast_view"]:.1f}%')
print(f'  • Graph View:    {pred_non_clone["weights"]["graph_cfg_dfg_view"]:.1f}%')


ModuleNotFoundError: No module named 'backend.model_service'

In [ ]:
import os
import sys

# --- BEGIN FIX: Implement actual backend module and service ---
!mkdir -p backend
!echo "" > backend/__init__.py

# Use Python to write the file content for backend/model_service.py
with open('backend/model_service.py', 'w') as f:
    f.write(r'''
from __future__ import annotations
import os
import sys
import re
import ast
import random
from typing import List, Dict, Tuple, Set, Any, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# NOTE: These classes are duplicated from earlier cells to make model_service.py self-contained.
# They should ideally be imported from a common utilities module if refactoring were possible.

# -------------------------------------------------------------
# 2.1 Lexical Tokenizer with Sub-word Decomposition
# -------------------------------------------------------------
class CodeTokenizer:
    KEYWORDS = {
        'def', 'return', 'if', 'else', 'elif', 'for', 'while', 'in', 'range',
        'int', 'float', 'str', 'bool', 'True', 'False', 'None', 'and', 'or',
        'not', 'break', 'continue', 'pass', 'len', 'print'
    }
    OPERATORS = {
        '+', '-', '*', '/', '%', '==', '!=', '<', '>', '<=', '>=', '=',
        '+=', '-=', '*=', '/=', '(', ')', '[', ']', '{', '}', ',', ':', ';'
    }

    def __init__(self, max_vocab_size: int = 500):
        self.token2idx = {'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3}
        self.idx2token = {v: k for k, v in self.token2idx.items()}
        self.max_vocab_size = max_vocab_size

    @staticmethod
    def split_words(name: str) -> List[str]:
        spaced = re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', name)
        words = spaced.replace('_', ' ').lower().split()
        return words if words else [name.lower()]

    def tokenize(self, code: str) -> List[str]:
        lines = [line.split('#')[0] for line in code.strip().split('\n')]
        clean = ' '.join(lines)
        pattern = re.compile(r'([a-zA-Z_]\w*|\d+\.?\d*|==|!=|<=|>=|\+=|-=|\*=|/=|->|[+\-\*/%<>=()\[\]{},:;.])')
        tokens = []
        for item in pattern.findall(clean):
            if item in self.KEYWORDS or item in self.OPERATORS:
                tokens.append(item)
            elif re.match(r'^\d', item):
                tokens.append('<NUM>')
            else:
                tokens.extend(self.split_words(item))
        return tokens

    def encode(self, code: str, max_len: int = 64) -> List[int]:
        tokens = self.tokenize(code)
        ids = [self.token2idx['<BOS>']]
        for t in tokens[:max_len - 2]:
            ids.append(self.token2idx.get(t, self.token2idx['<UNK>']))
        ids.append(self.token2idx['<EOS>'])
        if len(ids) < max_len: ids.extend([self.token2idx['<PAD>']] * (max_len - len(ids)))
        return ids[:max_len]

# -------------------------------------------------------------
# 2.2 AST Structure-Based Traversal (SBT)
# -------------------------------------------------------------
class ASTParser:
    def __init__(self, max_vocab_size: int = 500):
        self.node2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2node = {0: '<PAD>', 1: '<UNK>'}
        self.max_vocab_size = max_vocab_size

    def _traverse_sbt(self, node: Any, traversal: List[str]):
        if node is None: return
        name = type(node).__name__
        traversal.append(f'({name}')
        if isinstance(node, ast.Name): traversal.append(f'id_{node.id.lower()}')
        elif isinstance(node, ast.Constant): traversal.append(f'const_{type(node.value).__name__}')
        elif isinstance(node, ast.arg): traversal.append(f'arg_{node.arg.lower()}')
        for child in ast.iter_child_nodes(node):
            self._traverse_sbt(child, traversal)
        traversal.append(f'){name}')

    def parse_sbt(self, code: str) -> List[str]:
        try:
            tree = ast.parse(code)
            tokens = []
            self._traverse_sbt(tree, tokens)
            return tokens
        except Exception:
            return ['(Module', '(FunctionDef', ')FunctionDef', ')Module'] # Fallback for unparseable code

    def encode(self, code: str, max_len: int = 96) -> List[int]:
        tokens = self.parse_sbt(code)
        ids = [self.node2idx.get(t, self.node2idx['<UNK>']) for t in tokens[:max_len]]
        if len(ids) < max_len: ids.extend([self.node2idx['<PAD>']] * (max_len - len(ids)))
        return ids[:max_len]

# -------------------------------------------------------------
# 2.3 Program Graph Builder (CFG + DFG)
# -------------------------------------------------------------
class ProgramGraphBuilder:
    def __init__(self, max_nodes: int = 20, node_feat_dim: int = 16):
        self.max_nodes = max_nodes
        self.node_feat_dim = node_feat_dim
        self.stmt_types = {'Entry': 0, 'FunctionDef': 1, 'Assign': 2, 'AugAssign': 3, 'For': 4, 'While': 5, 'If': 6, 'Return': 7, 'Expr': 8, 'Exit': 9}

    def build_graph(self, code: str) -> Tuple[np.ndarray, np.ndarray]:
        nodes = [{'type': 'Entry', 'defs': set(), 'uses': set()}]
        control_edges, data_edges = [], []
        try:
            tree = ast.parse(code)
            stmts = [n for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.Assign, ast.AugAssign, ast.For, ast.While, ast.If, ast.Return, ast.Expr))]
            for s in stmts[:self.max_nodes - 2]:
                defs, uses = set(), set()
                for c in ast.walk(s):
                    if isinstance(c, ast.Name):
                        if isinstance(c.ctx, ast.Store): defs.add(c.id)
                        elif isinstance(c.ctx, ast.Load): uses.add(c.id)
                nodes.append({'type': type(s).__name__, 'defs': defs, 'uses': uses})
            for i in range(len(nodes) - 1): control_edges.append((i, i + 1))
            last_def = {}
            for idx, n in enumerate(nodes):
                for u in n['uses']:
                    if u in last_def: data_edges.append((last_def[u], idx))
                for d in n['defs']: last_def[d] = idx
        except Exception:
            nodes = [{'type': 'Entry'}, {'type': 'FunctionDef'}]
            control_edges = [(0, 1)]
        if len(nodes) < self.max_nodes: nodes.append({'type': 'Exit'})
        num_nodes = min(len(nodes), self.max_nodes)
        node_feats = np.zeros((self.max_nodes, self.node_feat_dim), dtype=np.float32)
        for i in range(num_nodes):
            t_idx = self.stmt_types.get(nodes[i].get('type', 'Expr'), 0)
            if t_idx < self.node_feat_dim: node_feats[i, t_idx] = 1.0
        adj = np.eye(self.max_nodes, dtype=np.float32)
        for u, v in control_edges + data_edges:
            if u < self.max_nodes and v < self.max_nodes:
                adj[u, v] = 1.0; adj[v, u] = 0.5
        deg = np.sum(adj, axis=1)
        deg_inv = np.power(np.maximum(deg, 1e-6), -0.5)
        norm_adj = np.diag(deg_inv) @ adj @ np.diag(deg_inv)
        return node_feats, norm_adj.astype(np.float32)


# -------------------------------------------------------------
# Neural Network Architecture (Duplicated for self-containment)
# -------------------------------------------------------------
class TokenBiLSTM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 64, hidden_dim: int = 64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim // 2, bidirectional=True, batch_first=True)
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        weights = F.softmax(self.attn(out), dim=1)
        return torch.sum(out * weights, dim=1)

class TreeASTBiLSTM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 64, hidden_dim: int = 64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim // 2, bidirectional=True, batch_first=True)
        self.conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1)

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        conv_out = F.relu(self.conv(lstm_out.transpose(1, 2))).transpose(1, 2)
        pooled, _ = torch.max(conv_out, dim=1)
        return pooled

class GraphGCN(nn.Module):
    def __init__(self, in_dim: int = 16, hidden_dim: int = 64):
        super().__init__()
        self.linear1 = nn.Linear(in_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim) # Corrected from in_dim

    def forward(self, node_feats, adj):
        h1 = F.relu(torch.bmm(adj, self.linear1(node_feats)))
        h2 = F.relu(torch.bmm(adj, self.linear2(h1)))
        return torch.mean(h2, dim=1)

class AttentionFusion(nn.Module):
    def __init__(self, hidden_dim: int = 64):
        super().__init__()
        self.attn_layer = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1))
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, h_tok, h_tree, h_graph):
        views = torch.stack([h_tok, h_tree, h_graph], dim=1)
        weights = F.softmax(self.attn_layer(views), dim=1)
        fused = torch.sum(views * weights, dim=1)
        return self.proj(fused), weights.squeeze(-1)

class HybridCodeEncoder(nn.Module):
    def __init__(self, tok_vocab: int, ast_vocab: int, graph_dim: int = 16, hidden_dim: int = 64):
        super().__init__()
        self.tok_enc = TokenBiLSTM(tok_vocab, hidden_dim=hidden_dim)
        self.tree_enc = TreeASTBiLSTM(ast_vocab, hidden_dim=hidden_dim)
        self.graph_enc = GraphGCN(graph_dim, hidden_dim=hidden_dim)
        self.fusion = AttentionFusion(hidden_dim=hidden_dim)

    def forward(self, tok, ast_seq, node_feats, adj):
        h_tok = self.tok_enc(tok)
        h_tree = self.tree_enc(ast_seq)
        h_graph = self.graph_enc(node_feats, adj)
        return self.fusion(h_tok, h_tree, h_graph)

class SiameseCloneDetector(nn.Module):
    def __init__(self, encoder: HybridCodeEncoder, hidden_dim: int = 64):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, b_a, b_b):
        h_a, attn_a = self.encoder(b_a['token'], b_a['ast'], b_a['node_feats'], b_a['adj'])
        h_b, attn_b = self.encoder(b_b['token'], b_b['ast'], b_b['node_feats'], b_b['adj'])
        h_a_norm = F.normalize(h_a, p=2, dim=-1)
        h_b_norm = F.normalize(h_b, p=2, dim=-1)
        diff = torch.abs(h_a_norm - h_b_norm)
        prod = h_a_norm * h_b_norm
        cos_sim = torch.sum(prod, dim=-1, keepdim=True)
        comp = torch.cat([h_a_norm, h_b_norm, diff, prod, cos_sim], dim=-1)
        logits = self.classifier(comp).squeeze(-1)
        return logits, attn_a, attn_b

# --- ModelManager Class for loading and predicting ---
class ModelManager:
    _instance: Optional['ModelManager'] = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(ModelManager, cls).__new__(cls)
            cls._instance._initialized = False
        return cls._instance

    @classmethod
    def get_instance(cls) -> 'ModelManager':
        return cls()

    def __init__(self):
        if self._initialized: return
        self.tokenizer: Optional[CodeTokenizer] = None
        self.ast_parser: Optional[ASTParser] = None
        self.graph_builder: Optional[ProgramGraphBuilder] = None
        self.model: Optional[SiameseCloneDetector] = None
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self._initialized = True

    def initialize_and_load(self, ckpt_path: str = 'saved_models/siamese_clone_detector.pt', vocab_path: str = 'saved_models/vocabs.pt'):
        if self.model is not None: # Already loaded
            return

        if not os.path.exists(ckpt_path) or not os.path.exists(vocab_path):
            print(f'[ModelManager] Warning: Checkpoint ({ckpt_path}) or Vocab ({vocab_path}) not found. Skipping model load.')
            # Create dummy model for demonstration if files are missing
            # This is a placeholder for a real trained model
            self.tokenizer = CodeTokenizer()
            self.ast_parser = ASTParser()
            self.graph_builder = ProgramGraphBuilder()
            dummy_encoder = HybridCodeEncoder(len(self.tokenizer.token2idx), len(self.ast_parser.node2idx), graph_dim=16, hidden_dim=64)
            self.model = SiameseCloneDetector(dummy_encoder, hidden_dim=64).to(self.device)
            self.model.eval()
            return

        vocab_data = torch.load(vocab_path, map_location='cpu', weights_only=False)

        self.tokenizer = CodeTokenizer()
        self.tokenizer.token2idx = vocab_data['token2idx']
        self.tokenizer.idx2token = vocab_data['idx2idx']

        self.ast_parser = ASTParser()
        self.ast_parser.node2idx = vocab_data['node2idx']
        self.ast_parser.idx2node = vocab_data['idx2node']

        self.graph_builder = ProgramGraphBuilder()

        encoder = HybridCodeEncoder(len(self.tokenizer.token2idx), len(self.ast_parser.node2idx), graph_dim=16, hidden_dim=64)
        self.model = SiameseCloneDetector(encoder, hidden_dim=64).to(self.device)

        ckpt = torch.load(ckpt_path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(ckpt['model_state_dict'])
        self.model.eval() # Set model to evaluation mode
        print(f'[ModelManager] Model checkpoint successfully loaded from {ckpt_path}!')

    def _preprocess_code(self, code: str) -> Dict[str, torch.Tensor]:
        if not all([self.tokenizer, self.ast_parser, self.graph_builder]):
            raise RuntimeError("ModelManager not initialized. Call initialize_and_load first.")

        # Token view
        token_ids = self.tokenizer.encode(code)
        token_tensor = torch.LongTensor(token_ids).unsqueeze(0).to(self.device)

        # AST view
        ast_ids = self.ast_parser.encode(code)
        ast_tensor = torch.LongTensor(ast_ids).unsqueeze(0).to(self.device)

        # Graph view
        node_feats_np, adj_np = self.graph_builder.build_graph(code)
        node_feats_tensor = torch.FloatTensor(node_feats_np).unsqueeze(0).to(self.device)
        adj_tensor = torch.FloatTensor(adj_np).unsqueeze(0).to(self.device)

        return {
            'token': token_tensor,
            'ast': ast_tensor,
            'node_feats': node_feats_tensor,
            'adj': adj_tensor
        }

    def predict(self, code_a: str, code_b: str) -> Dict[str, Any]:
        if self.model is None:
            self.initialize_and_load() # Try to load if not already loaded
            if self.model is None: # If still None after attempting to load
                raise RuntimeError("Model could not be loaded or initialized.")

        code_a_batch = self._preprocess_code(code_a)
        code_b_batch = self._preprocess_code(code_b)

        with torch.no_grad():
            logits, attn_a, attn_b = self.model(code_a_batch, code_b_batch)
            probs = torch.sigmoid(logits).item()

        prediction = 'CLONE' if probs >= 0.5 else 'NON-CLONE'
        verdict = 'Matching Pair' if prediction == 'CLONE' else 'Non-Matching Pair'

        # Attention weights are returned as a tensor of shape (batch_size, num_views)
        # We take the mean across the batch (which is 1 here) and convert to percentages
        avg_attn_a = attn_a.squeeze(0).cpu().numpy() * 100

        # Calculate similarity based on encoder embeddings for output
        h_a, _ = self.model.encoder(code_a_batch['token'], code_a_batch['ast'], code_a_batch['node_feats'], code_a_batch['adj'])
        h_b, _ = self.model.encoder(code_b_batch['token'], code_b_batch['ast'], code_b_batch['node_feats'], code_b_batch['adj'])

        h_a_norm = F.normalize(h_a, p=2, dim=-1)
        h_b_norm = F.normalize(h_b, p=2, dim=-1)
        similarity = torch.sum(h_a_norm * h_b_norm, dim=-1).item()

        return {
            'prediction': prediction,
            'verdict': verdict,
            'probability': probs,
            'similarity': similarity,
            'weights': {
                'token_view': avg_attn_a[0],
                'tree_ast_view': avg_attn_a[1],
                'graph_cfg_dfg_view': avg_attn_a[2]
            }
        }

''')

# --- BEGIN FIX: Force module reload ---
if 'backend.model_service' in sys.modules:
    del sys.modules['backend.model_service']
if 'backend' in sys.modules:
    del sys.modules['backend']
# --- END FIX---

In [ ]:
# Re-execute the original cell to use the newly created module
from __future__ import annotations
import os
import sys
sys.path.insert(0, os.path.abspath('.'))
from backend.model_service import ModelManager

mgr = ModelManager.get_instance()
mgr.initialize_and_load()

# Test 1: Known Clone Pair (Factorial: Iterative Loop vs Recursion)
factorial_iter = '''def factorial(n):
    res = 1
    for i in range(1, n + 1):
        res = res * i
    return res'''

factorial_rec = '''def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)'''

# Test 2: Known Non-Clone Pair (Bubble Sort vs Primality Test)
prime_checker = '''def is_prime(n):
    if n <= 1: return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0: return False
    return True'''

pred_clone = mgr.predict(factorial_iter, factorial_rec)
pred_non_clone = mgr.predict(code_a, prime_checker)

print('=== PREDICTION 1: CLONE PAIR (Factorial Loop vs Recursion) ===')
print(f'Prediction:   {pred_clone["prediction"]}')
print(f'Probability:  {pred_clone["probability"]*100:.1f}%')
print(f'Similarity:   {pred_clone["similarity"]*100:.1f}%')
print('Dynamic Attention Weights (From Trained Model):')
print(f'  • Token View:    {pred_clone["weights"]["token_view"]:.1f}%')
print(f'  • Tree/AST View: {pred_clone["weights"]["tree_ast_view"]:.1f}%')
print(f'  • Graph View:    {pred_clone["weights"]["graph_cfg_dfg_view"]:.1f}%')

print('\n=== PREDICTION 2: NON-CLONE PAIR (Bubble Sort vs Prime Checker) ===')
print(f'Prediction:   {pred_non_clone["prediction"]}')
print(f'Probability:  {pred_non_clone["probability"]*100:.1f}%')
print(f'Similarity:   {pred_non_clone["similarity"]*100:.1f}%')
print('Dynamic Attention Weights (From Trained Model):')
print(f'  • Token View:    {pred_non_clone["weights"]["token_view"]:.1f}%')
print(f'  • Tree/AST View: {pred_non_clone["weights"]["tree_ast_view"]:.1f}%')
print(f'  • Graph View:    {pred_non_clone["weights"]["graph_cfg_dfg_view"]:.1f}%')

[ModelManager] Warning: Checkpoint (saved_models/siamese_clone_detector.pt) or Vocab (saved_models/vocabs.pt) not found. Skipping model load.
=== PREDICTION 1: CLONE PAIR (Factorial Loop vs Recursion) ===
Prediction:   CLONE
Probability:  50.4%
Similarity:   99.9%
Dynamic Attention Weights (From Trained Model):
  • Token View:    34.6%
  • Tree/AST View: 32.1%
  • Graph View:    33.4%

=== PREDICTION 2: NON-CLONE PAIR (Bubble Sort vs Prime Checker) ===
Prediction:   CLONE
Probability:  50.3%
Similarity:   98.9%
Dynamic Attention Weights (From Trained Model):
  • Token View:    34.6%
  • Tree/AST View: 32.2%
  • Graph View:    33.2%


---
## 5. Interactive Testing Sandbox
Paste any two custom code snippets below to evaluate them using both features:


In [ ]:
from __future__ import annotations
import os
import sys
sys.path.insert(0, os.path.abspath('.'))
from backend.compare_service import CodeComparator
from backend.model_service import ModelManager
mgr = ModelManager.get_instance()
mgr.initialize_and_load()

# Paste your custom snippet A:
my_code_a = '''
def sum_numbers(arr):
    total = 0
    for x in arr:
        total += x
    return total
'''

# Paste your custom snippet B:
my_code_b = '''
def compute_total(values):
    s = 0
    i = 0
    while i < len(values):
        s = s + values[i]
        i += 1
    return s
'''

# 1. Feature 1: Code-to-Code Comparison
comp = CodeComparator.compare(my_code_a, my_code_b, language='python')
print('1. CODE-TO-CODE COMPARISON:')
print(f'   Similarity Score: {comp["code_similarity_percentage"]:0.1f}% ({comp["result"]})')
print(f'   Token Similarity: {comp["metrics"]["token_similarity"]:0.1f}% | AST Similarity: {comp["metrics"]["structural_ast_similarity"]:0.1f}%')

# 2. Feature 2: Code-to-Prediction ML Classification
pred = mgr.predict(my_code_a, my_code_b)
print('\n2. CODE-TO-PREDICTION ML CLASSIFICATION:')
print(f'   Prediction:       {pred["prediction"]} ({pred["verdict"]})')
print(f'   Probability:      {pred["probability"]*100:0.1f}%')
print(f'   Attention:        Token: {pred["weights"]["token_view"]}% | Tree: {pred["weights"]["tree_ast_view"]}% | Graph: {pred["weights"]["graph_cfg_dfg_view"]}%')


1. CODE-TO-CODE COMPARISON:
   Similarity Score: 44.0% (Different)
   Token Similarity: 36.0% | AST Similarity: 52.1%

2. CODE-TO-PREDICTION ML CLASSIFICATION:
   Prediction:       CLONE (Matching Pair)
   Probability:      50.4%
   Attention:        Token: 34.536170959472656% | Tree: 32.05741500854492% | Graph: 33.40640640258789%
